# Appendix B.1 — complete used-car bargaining data analysis

This notebook is a self-contained analysis pipeline for Appendix B.1 of the manuscript. It uses the uploaded `0421_redo_previous_exp.rar` archive, extracts the six fixed-policy randomized-buyer experiments, rebuilds the processed turn-level dataset, and then runs the feature/model comparison and validation analyses described in the appendix.

The pipeline intentionally recomputes all counts rather than hard-coding manuscript values. In the uploaded archive, each fixed policy has **30 runs per vehicle**; the draft text currently describes a larger 100-run version. The table proportions should still be directly comparable, while absolute pooled-turn counts scale with the number of runs.

Main outputs are written under `appendix_B1_outputs/`:

- `tables/*.csv` and `tables/*.tex`
- `figures/*.png`
- `predictions/*.csv`

## 0. Configuration

Set `FULL_APPENDIX_RUN = True` for the complete 10-fold grid over all feature maps and model classes. During development, set it to `False` to run a shorter smoke test.

In [2]:
from __future__ import annotations

import ctypes
import ctypes.util
import json
import math
import os
import random
import shutil
import stat
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------
# Run controls
# ---------------------------
FULL_APPENDIX_RUN = True     # complete Appendix B.1 grid
INCLUDE_MLP = True           # set False if the full grid is too slow
OUTER_FOLDS = 10 if FULL_APPENDIX_RUN else 5
INNER_FOLDS = 3              # grouped inner CV for Ridge/GAM alpha selection
RANDOM_SEED = 20260421

# In smoke-test mode we only run a smaller feature/model grid.
SMOKE_FEATURES = ["II", "III", "IX", "X"]
SMOKE_MODELS = ["Base", "OLS", "Ridge", "GBR"]

# Numerical constants from the Appendix B.1 specification.
ETA = 1e-4
AGREEMENT_MULTIPLIER = 1.01
MODEL_ROUND_MIN = 3
MODEL_ROUND_MAX = 12
RIDGE_ALPHAS = np.array([1e-3, 1e-2, 1e-1, 1.0, 10.0])

# Paths. The notebook is designed to work when placed next to the uploaded RAR.
HERE = Path.cwd()
ARCHIVE_PATH = HERE / "0421_redo_previous_exp.rar"
if not ARCHIVE_PATH.exists():
    # Fallback for the ChatGPT sandbox path used while preparing this notebook.
    sandbox_archive = Path("../0421_redo_previous_exp.rar")
    if sandbox_archive.exists():
        ARCHIVE_PATH = sandbox_archive

EXTRACT_DIR = HERE / "_extracted_0421_redo_previous_exp"
RESULTS_DIR = EXTRACT_DIR / "0421_redo_previous_exp" / "results"

OUTPUT_DIR = HERE / "appendix_B1_outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
PRED_DIR = OUTPUT_DIR / "predictions"
for d in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("Archive:", ARCHIVE_PATH)
print("Extraction target:", EXTRACT_DIR)
print("Output directory:", OUTPUT_DIR)
print("FULL_APPENDIX_RUN:", FULL_APPENDIX_RUN, "INCLUDE_MLP:", INCLUDE_MLP)

Archive: ..\0421_redo_previous_exp.rar
Extraction target: d:\CODING\LLMNego_Plus\EXP\0424_main_exp_data_analysis\_extracted_0421_redo_previous_exp
Output directory: d:\CODING\LLMNego_Plus\EXP\0424_main_exp_data_analysis\appendix_B1_outputs
FULL_APPENDIX_RUN: True INCLUDE_MLP: True


## 1. Extract and load the uploaded experiment archive

In [3]:
def _safe_join(base: Path, member_name: str) -> Path:
    """Return a safe extraction path under `base`."""
    dest = (base / member_name).resolve()
    base_resolved = base.resolve()
    if not str(dest).startswith(str(base_resolved)):
        raise RuntimeError(f"Unsafe archive path: {member_name}")
    return dest


def extract_rar_with_libarchive(archive_path: Path, out_dir: Path) -> int:
    """Extract a RAR archive using the system libarchive shared library.

    This avoids relying on a shell `unrar` binary. It worked for the uploaded
    RAR5 archive in the preparation environment.
    """
    libpath = ctypes.util.find_library("archive") or "/lib/x86_64-linux-gnu/libarchive.so.13"
    lib = ctypes.CDLL(libpath)

    c_void_p = ctypes.c_void_p
    c_char_p = ctypes.c_char_p
    c_size_t = ctypes.c_size_t
    c_ssize_t = getattr(ctypes, "c_ssize_t", ctypes.c_long)

    lib.archive_read_new.restype = c_void_p
    lib.archive_read_support_filter_all.argtypes = [c_void_p]
    lib.archive_read_support_filter_all.restype = ctypes.c_int
    lib.archive_read_support_format_all.argtypes = [c_void_p]
    lib.archive_read_support_format_all.restype = ctypes.c_int
    lib.archive_read_open_filename.argtypes = [c_void_p, c_char_p, c_size_t]
    lib.archive_read_open_filename.restype = ctypes.c_int
    lib.archive_read_next_header.argtypes = [c_void_p, ctypes.POINTER(c_void_p)]
    lib.archive_read_next_header.restype = ctypes.c_int
    lib.archive_entry_pathname.argtypes = [c_void_p]
    lib.archive_entry_pathname.restype = c_char_p
    lib.archive_entry_size.argtypes = [c_void_p]
    lib.archive_entry_size.restype = ctypes.c_longlong
    lib.archive_entry_filetype.argtypes = [c_void_p]
    lib.archive_entry_filetype.restype = ctypes.c_uint
    lib.archive_read_data.argtypes = [c_void_p, c_void_p, c_size_t]
    lib.archive_read_data.restype = c_ssize_t
    lib.archive_read_data_skip.argtypes = [c_void_p]
    lib.archive_read_data_skip.restype = ctypes.c_int
    lib.archive_error_string.argtypes = [c_void_p]
    lib.archive_error_string.restype = c_char_p
    lib.archive_read_free.argtypes = [c_void_p]
    lib.archive_read_free.restype = ctypes.c_int

    ARCHIVE_OK = 0
    ARCHIVE_EOF = 1

    out_dir.mkdir(parents=True, exist_ok=True)
    a = lib.archive_read_new()

    def err() -> str:
        s = lib.archive_error_string(a)
        return s.decode("utf-8", errors="replace") if s else ""

    try:
        for setup_fn in [lib.archive_read_support_filter_all, lib.archive_read_support_format_all]:
            r = setup_fn(a)
            if r != ARCHIVE_OK:
                raise RuntimeError(f"libarchive setup failed: code={r} {err()}")

        r = lib.archive_read_open_filename(a, os.fsencode(str(archive_path)), 10240)
        if r != ARCHIVE_OK:
            raise RuntimeError(f"libarchive open failed: code={r} {err()}")

        entry = c_void_p()
        n_entries = 0
        while True:
            r = lib.archive_read_next_header(a, ctypes.byref(entry))
            if r == ARCHIVE_EOF:
                break
            if r != ARCHIVE_OK:
                raise RuntimeError(f"libarchive next-header failed: code={r} {err()}")

            name = lib.archive_entry_pathname(entry).decode("utf-8", errors="replace")
            dest = _safe_join(out_dir, name)
            filetype = lib.archive_entry_filetype(entry)

            if name.endswith("/") or filetype == stat.S_IFDIR:
                dest.mkdir(parents=True, exist_ok=True)
                lib.archive_read_data_skip(a)
            else:
                dest.parent.mkdir(parents=True, exist_ok=True)
                with open(dest, "wb") as fh:
                    buf = ctypes.create_string_buffer(1024 * 1024)
                    while True:
                        n = lib.archive_read_data(a, buf, len(buf))
                        if n == 0:
                            break
                        if n < 0:
                            raise RuntimeError(f"libarchive read failed for {name}: code={n} {err()}")
                        fh.write(buf.raw[:n])
            n_entries += 1
        return n_entries
    finally:
        lib.archive_read_free(a)


def extract_archive_if_needed(archive_path: Path, extract_dir: Path, results_dir: Path) -> None:
    if results_dir.exists() and list(results_dir.glob("*.json")):
        print("Already extracted:", results_dir)
        return

    if not archive_path.exists():
        raise FileNotFoundError(
            f"Could not find {archive_path}. Put 0421_redo_previous_exp.rar next to this notebook "
            "or edit ARCHIVE_PATH in the configuration cell."
        )

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # First try Python rarfile if the host has an unrar backend; then use libarchive.
    try:
        import rarfile  # type: ignore
        with rarfile.RarFile(str(archive_path)) as rf:
            rf.extractall(str(extract_dir))
        print("Extracted with rarfile:", results_dir)
        return
    except Exception as e:
        print("rarfile extraction did not work; trying libarchive instead.")
        print("rarfile error:", type(e).__name__, str(e)[:200])

    n = extract_rar_with_libarchive(archive_path, extract_dir)
    print(f"Extracted {n} archive entries with libarchive.")
    if not results_dir.exists():
        raise RuntimeError(f"Extraction finished, but results directory was not found: {results_dir}")


extract_archive_if_needed(ARCHIVE_PATH, EXTRACT_DIR, RESULTS_DIR)
print("Result files:", len(list(RESULTS_DIR.glob('*.json'))))
print([p.name for p in sorted(RESULTS_DIR.glob('*.json'))[:6]])

rarfile extraction did not work; trying libarchive instead.
rarfile error: ModuleNotFoundError No module named 'rarfile'


FileNotFoundError: Could not find module 'd:\lib\x86_64-linux-gnu\libarchive.so.13' (or one of its dependencies). Try using the full path with constructor syntax.

## 2. Flatten raw JSON logs into a turn-level table

The six fixed-policy experiments are the randomized-buyer data used in Appendix B.1. The archive also contains transition experiments, but those are not included in this fixed-policy analysis.

In [ ]:
POLICY_KEY_TO_LABEL = {
    "patient_value_defender": "PVD",
    "busy_impatient_closer": "IC",
    "friendly_rapport_builder": "FRB",
    "reciprocal_fairness_keeper": "RFK",
    "opponent_aware_diagnostic": "OADS",
    "market_expert_value_justifier": "MEVJ",
}
POLICY_LABEL_TO_KEY = {v: k for k, v in POLICY_KEY_TO_LABEL.items()}
POLICY_ORDER = ["PVD", "IC", "FRB", "RFK", "OADS", "MEVJ"]


def _policy_label_from_key(key: str) -> str:
    return POLICY_KEY_TO_LABEL.get(key, key)


def load_fixed_policy_turns(results_dir: Path) -> tuple[pd.DataFrame, list[str]]:
    """Load the six fixed-policy result JSON files and return one row per seller turn."""
    files = sorted(results_dir.glob("exp0[0-5]_fixed_*.json"))
    if len(files) != 6:
        raise RuntimeError(f"Expected 6 fixed-policy JSON files, found {len(files)} in {results_dir}")

    rows: list[dict] = []
    fleet_order: list[str] = []

    for file in files:
        with open(file, "r", encoding="utf-8") as fh:
            data = json.load(fh)

        if not fleet_order and data.get("fleet"):
            fleet_order = [c["car_name"] for c in data["fleet"]]

        exp = data["experiment"]
        tag = exp["tag"]
        policy_key = tag.split("_fixed_", 1)[1]
        policy = _policy_label_from_key(policy_key)

        for run_index, run in enumerate(data["runs"]):
            car = run["car"]
            car_name = car["car_name"]
            for rr in run["rounds"]:
                rows.append(
                    {
                        "source_file": file.name,
                        "experiment_tag": tag,
                        "policy_key": policy_key,
                        "policy": policy,
                        "run_index_in_file": run_index,
                        "run_id": f"{tag}::run{run_index:04d}::seed{run.get('rng_seed')}",
                        "rng_seed": run.get("rng_seed"),
                        "car_name": car_name,
                        "M": float(car["market_price_new"]),
                        "v": float(car["dealer_cost"]),
                        "round": int(rr["round"]),
                        "buyer_price_raw": rr.get("buyer_price"),
                        "seller_price_raw": rr.get("seller_price"),
                        "buyer_template_id": rr.get("buyer_template_id"),
                        "seller_active_strategy": rr.get("seller_active_strategy"),
                        "seller_strategy_changed": rr.get("seller_strategy_changed"),
                    }
                )

    raw = pd.DataFrame(rows)
    if not fleet_order:
        fleet_order = list(raw["car_name"].drop_duplicates())

    vehicle_map = {name: f"V{i+1}" for i, name in enumerate(fleet_order)}
    raw["vehicle"] = raw["car_name"].map(vehicle_map)
    raw["policy"] = pd.Categorical(raw["policy"], categories=POLICY_ORDER, ordered=True)
    raw["vehicle"] = pd.Categorical(raw["vehicle"], categories=[f"V{i}" for i in range(1, 6)], ordered=True)
    return raw, fleet_order


raw_turns, fleet_order = load_fixed_policy_turns(RESULTS_DIR)

print("Raw rows:", len(raw_turns))
print("Policies:", list(raw_turns["policy"].cat.categories))
print("Vehicles:")
for i, name in enumerate(fleet_order, start=1):
    spec = raw_turns.loc[raw_turns["car_name"].eq(name), ["M", "v"]].iloc[0]
    print(f"  V{i}: {name:35s}  M={spec.M:,.0f}  v={spec.v:,.0f}  v/M={spec.v/spec.M:.3f}")

runs_per_cell = raw_turns.drop_duplicates(["policy", "vehicle", "run_id"]).groupby(["policy", "vehicle"], observed=True).size().unstack()
runs_per_cell

## 3. Preprocessing exactly as in Appendix B.1

For each turn:

1. Parse and retain the buyer bid \(B_t\) and seller proposal \(S_t\).
2. Truncate seller proposals to the rational interval \([v,M]\).
3. Treat a turn as an agreement if the truncated seller proposal satisfies \(S_t \le 1.01 B_t\).
4. Discard only observations **after** the first agreement turn in the same run.
5. Normalize prices by \((x-v)/(M-v)\).
6. Build lag, spread, and concession features for later model fitting.

Rows with unparsed seller prices are excluded from target rows. They are still retained while constructing lag features so that a missing previous seller price produces a missing lag rather than silently skipping over the failed round.

In [ ]:
def preprocess_turns(raw: pd.DataFrame, agreement_multiplier: float = AGREEMENT_MULTIPLIER) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (all_raw_marked, lagged_turns_before_after-agreement, clean_turns)."""
    out = raw.copy()
    out["B"] = pd.to_numeric(out["buyer_price_raw"], errors="coerce")
    out["S_raw"] = pd.to_numeric(out["seller_price_raw"], errors="coerce")
    out["parse_fail"] = out["S_raw"].isna()

    out["lower_truncated"] = out["S_raw"] < out["v"]
    out["upper_truncated"] = out["S_raw"] > out["M"]
    out["S"] = out["S_raw"].clip(lower=out["v"], upper=out["M"])

    out["agreement_flag"] = (~out["parse_fail"]) & (out["S"] <= agreement_multiplier * out["B"])
    first_agreement = (
        out.loc[out["agreement_flag"]]
        .groupby("run_id", observed=True)["round"]
        .min()
        .rename("first_agreement_round")
    )
    out = out.merge(first_agreement, on="run_id", how="left")
    out["after_agreement"] = out["first_agreement_round"].notna() & (out["round"] > out["first_agreement_round"])

    # Keep parse failures for lag construction, but never as targets.
    turns = out.loc[~out["after_agreement"]].copy()
    turns["margin"] = turns["M"] - turns["v"]
    turns["B_norm"] = (turns["B"] - turns["v"]) / turns["margin"]
    turns["S_norm"] = (turns["S"] - turns["v"]) / turns["margin"]

    turns = turns.sort_values(["policy", "vehicle", "run_id", "round"]).reset_index(drop=True)
    g = turns.groupby("run_id", sort=False, observed=True)

    for k in [1, 2]:
        turns[f"B_lag{k}"] = g["B_norm"].shift(k)
        turns[f"S_lag{k}"] = g["S_norm"].shift(k)
        turns[f"round_lag{k}"] = g["round"].shift(k)

    turns["dB"] = turns["B_norm"] - turns["B_lag1"]
    turns["dS_lag1"] = turns["S_lag1"] - turns["S_lag2"]
    turns["A"] = turns["S_lag1"] - turns["B_norm"]
    turns["A_lag1"] = turns["S_lag2"] - turns["B_lag1"]

    for col in ["B_norm", "B_lag1", "dB", "dS_lag1"]:
        turns[f"{col}_pos"] = np.maximum(turns[col], 0.0)
        turns[f"{col}_neg"] = np.maximum(-turns[col], 0.0)

    clean_turns = turns.loc[~turns["parse_fail"]].copy()
    return out, turns, clean_turns


all_marked, lagged_turns, clean_turns = preprocess_turns(raw_turns)

print("Raw turns:", len(raw_turns))
print("Removed after agreement:", int(all_marked["after_agreement"].sum()))
print("Parse failures before/at agreement:", int((~all_marked["after_agreement"] & all_marked["parse_fail"]).sum()))
print("Clean turns for descriptive tables:", len(clean_turns))

parse_fail_by_policy = (
    all_marked.groupby("policy", observed=True)["parse_fail"]
    .sum()
    .reindex(POLICY_ORDER)
    .rename("parse_failures")
)
parse_fail_by_policy

## 4. Appendix tables: pooled turns and truncation rates

In [ ]:
def save_table(df: pd.DataFrame, stem: str, float_format: str | None = None) -> None:
    csv_path = TABLE_DIR / f"{stem}.csv"
    tex_path = TABLE_DIR / f"{stem}.tex"
    df.to_csv(csv_path)
    df.to_latex(tex_path, float_format=float_format)
    print("wrote", csv_path)
    print("wrote", tex_path)


table_turn_counts = (
    clean_turns.groupby(["policy", "vehicle"], observed=True)
    .size()
    .unstack()
    .reindex(index=POLICY_ORDER, columns=[f"V{i}" for i in range(1, 6)])
)
table_turn_counts["Total"] = table_turn_counts.sum(axis=1)
table_turn_counts.loc["Total"] = table_turn_counts.sum(axis=0)
table_turn_counts = table_turn_counts.astype(int)

table_lower_trunc = (
    100
    * clean_turns.groupby(["policy", "vehicle"], observed=True)["lower_truncated"]
    .mean()
    .unstack()
    .reindex(index=POLICY_ORDER, columns=[f"V{i}" for i in range(1, 6)])
).round(1)

table_upper_trunc = (
    100
    * clean_turns.groupby(["policy", "vehicle"], observed=True)["upper_truncated"]
    .mean()
    .unstack()
    .reindex(index=POLICY_ORDER, columns=[f"V{i}" for i in range(1, 6)])
).round(1)

save_table(table_turn_counts, "B1_table_turn_counts")
save_table(table_lower_trunc, "B1_table_lower_truncation_pct", float_format="%.1f")
save_table(table_upper_trunc, "B1_table_upper_truncation_pct", float_format="%.1f")

display(table_turn_counts)
display(table_lower_trunc)
display(table_upper_trunc)

## 5. Candidate feature maps

The candidate feature maps I–X follow the Appendix B.1 notation. The intercept is not included as a column because scikit-learn estimators fit it separately; `dim_phi` below adds it back when reporting feature dimension.

In [ ]:
FEATURE_SPECS = OrderedDict(
    [
        ("I", {
            "description": r"(1, B_t)",
            "cols": ["B_norm"],
            "lag_order": 0,
        }),
        ("II", {
            "description": r"(1, B_t, S_{t-1}, B_{t-1})",
            "cols": ["B_norm", "S_lag1", "B_lag1"],
            "lag_order": 1,
        }),
        ("III", {
            "description": r"(1, B_t, S_{t-1}, B_{t-1}, S_{t-2}, B_{t-2})",
            "cols": ["B_norm", "S_lag1", "B_lag1", "S_lag2", "B_lag2"],
            "lag_order": 2,
        }),
        ("IV", {
            "description": r"(1, A_t, S_{t-1})",
            "cols": ["A", "S_lag1"],
            "lag_order": 1,
        }),
        ("V", {
            "description": r"(1, A_t, S_{t-1}, A_{t-1}, S_{t-2})",
            "cols": ["A", "S_lag1", "A_lag1", "S_lag2"],
            "lag_order": 2,
        }),
        ("VI", {
            "description": r"(1, B_t^+, B_t^-)",
            "cols": ["B_norm_pos", "B_norm_neg"],
            "lag_order": 0,
        }),
        ("VII", {
            "description": r"(1, B_t^+, B_t^-, S_{t-1}, B_{t-1}^+, B_{t-1}^-)",
            "cols": ["B_norm_pos", "B_norm_neg", "S_lag1", "B_lag1_pos", "B_lag1_neg"],
            "lag_order": 1,
        }),
        ("VIII", {
            "description": r"(1, B_t, S_{t-1}, ΔB_t)",
            "cols": ["B_norm", "S_lag1", "dB"],
            "lag_order": 1,
        }),
        ("IX", {
            "description": r"(1, B_t^+, B_t^-, S_{t-1}, ΔB_t^+, ΔB_t^-)",
            "cols": ["B_norm_pos", "B_norm_neg", "S_lag1", "dB_pos", "dB_neg"],
            "lag_order": 1,
        }),
        ("X", {
            "description": r"(1, B_t^+, B_t^-, S_{t-1}, ΔB_t^+, ΔB_t^-, ΔS_{t-1}^+, ΔS_{t-1}^-)",
            "cols": ["B_norm_pos", "B_norm_neg", "S_lag1", "dB_pos", "dB_neg", "dS_lag1_pos", "dS_lag1_neg"],
            "lag_order": 2,
        }),
    ]
)

BASELINE_SPEC = {
    "description": r"Base: θ0 + θ1 S_{t-1} on normalized scale",
    "cols": ["S_lag1"],
    "lag_order": 1,
}

feature_summary = pd.DataFrame(
    [
        {
            "feature": name,
            "description": spec["description"],
            "lag_order": spec["lag_order"],
            "dim_phi": len(spec["cols"]) + 1,
            "columns": ", ".join(spec["cols"]),
        }
        for name, spec in FEATURE_SPECS.items()
    ]
)
display(feature_summary)

## 6. Model and grouped-CV helpers

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40, 40)))


def logit_clip(p: np.ndarray, eta: float = ETA) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), eta, 1.0 - eta)
    return np.log(p / (1.0 - p))


def make_group_stratified_folds(
    groups: np.ndarray,
    strata: np.ndarray | None = None,
    n_splits: int = 10,
    seed: int = RANDOM_SEED,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Return train/test index arrays with whole runs assigned to one fold.

    The assignment is stratified by vehicle so that each fold contains a
    roughly balanced set of vehicles.
    """
    groups = np.asarray(groups)
    if strata is None:
        strata = np.repeat("all", len(groups))
    strata = np.asarray(strata)

    group_df = pd.DataFrame({"group": groups, "stratum": strata}).drop_duplicates("group")
    n_unique_groups = group_df["group"].nunique()
    n_splits = min(n_splits, n_unique_groups)
    if n_splits < 2:
        raise ValueError("Need at least two groups for grouped CV.")

    rng = np.random.default_rng(seed)
    fold_groups = [[] for _ in range(n_splits)]

    for _, block in group_df.groupby("stratum", sort=False):
        gs = block["group"].to_numpy().copy()
        rng.shuffle(gs)
        for i, g in enumerate(gs):
            fold_groups[i % n_splits].append(g)

    all_idx = np.arange(len(groups))
    folds: list[tuple[np.ndarray, np.ndarray]] = []
    for k in range(n_splits):
        test_group_set = set(fold_groups[k])
        test_mask = np.array([g in test_group_set for g in groups])
        folds.append((all_idx[~test_mask], all_idx[test_mask]))
    return folds


def build_model_frame(
    turns: pd.DataFrame,
    policy: str,
    feature_name: str,
    round_min: int = MODEL_ROUND_MIN,
    round_max: int = MODEL_ROUND_MAX,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, list[str]]:
    """Build X/y/meta for one policy and one feature specification."""
    if feature_name == "Base":
        spec = BASELINE_SPEC
    else:
        spec = FEATURE_SPECS[feature_name]

    cols = list(spec["cols"])
    lag_order = int(spec["lag_order"])

    sub = turns.loc[
        (turns["policy"].astype(str) == policy)
        & turns["round"].between(round_min, round_max)
        & (~turns["parse_fail"])
    ].copy()

    required = cols + ["S_norm", "margin", "run_id", "vehicle", "round"]
    if lag_order >= 1:
        required += ["round_lag1"]
    if lag_order >= 2:
        required += ["round_lag2"]

    mask = sub[required].notna().all(axis=1)
    if lag_order >= 1:
        mask &= sub["round_lag1"].eq(sub["round"] - 1)
    if lag_order >= 2:
        mask &= sub["round_lag2"].eq(sub["round"] - 2)

    sub = sub.loc[mask].copy()
    X = sub[cols].to_numpy(dtype=float)
    y = sub["S_norm"].to_numpy(dtype=float)
    return sub, X, y, cols


def select_alpha_grouped(
    model_kind: str,
    X: np.ndarray,
    y_norm: np.ndarray,
    groups: np.ndarray,
    strata: np.ndarray,
    alphas: np.ndarray = RIDGE_ALPHAS,
    inner_folds: int = INNER_FOLDS,
    seed: int = RANDOM_SEED,
) -> float:
    """Choose Ridge/GAM penalty by grouped inner CV on normalized-scale MAE."""
    y_logit = logit_clip(y_norm)
    folds = make_group_stratified_folds(groups, strata, n_splits=inner_folds, seed=seed)
    scores: list[tuple[float, float]] = []

    for alpha in alphas:
        fold_mae = []
        for tr, va in folds:
            if model_kind == "Ridge":
                estimator = Pipeline(
                    [
                        ("scale", StandardScaler()),
                        ("model", Ridge(alpha=float(alpha))),
                    ]
                )
            elif model_kind == "GAM":
                estimator = Pipeline(
                    [
                        ("scale", StandardScaler()),
                        ("spline", SplineTransformer(n_knots=6, degree=3, include_bias=False)),
                        ("model", Ridge(alpha=float(alpha))),
                    ]
                )
            else:
                raise ValueError(f"Unknown alpha-tuned model: {model_kind}")

            estimator.fit(X[tr], y_logit[tr])
            pred = sigmoid(estimator.predict(X[va]))
            fold_mae.append(np.mean(np.abs(pred - y_norm[va])))

        scores.append((float(alpha), float(np.mean(fold_mae))))

    return min(scores, key=lambda x: x[1])[0]


def fit_estimator(
    model_name: str,
    X_train: np.ndarray,
    y_train_norm: np.ndarray,
    *,
    groups_train: np.ndarray | None = None,
    strata_train: np.ndarray | None = None,
    seed: int = RANDOM_SEED,
    fixed_alpha: float | None = None,
):
    """Fit one model. Non-baseline models are trained on the logit target."""
    if model_name == "Base":
        model = LinearRegression()
        model.fit(X_train, y_train_norm)
        return model, {"target_scale": "normalized", "alpha": np.nan}

    y_train_logit = logit_clip(y_train_norm)

    if model_name == "OLS":
        model = Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", LinearRegression()),
            ]
        )
        model.fit(X_train, y_train_logit)
        return model, {"target_scale": "logit", "alpha": np.nan}

    if model_name in {"Ridge", "GAM"}:
        if fixed_alpha is None:
            if groups_train is None:
                groups_train = np.arange(len(y_train_norm))
            if strata_train is None:
                strata_train = np.repeat("all", len(y_train_norm))
            alpha = select_alpha_grouped(
                model_name,
                X_train,
                y_train_norm,
                groups_train,
                strata_train,
                seed=seed,
            )
        else:
            alpha = float(fixed_alpha)

        if model_name == "Ridge":
            model = Pipeline(
                [
                    ("scale", StandardScaler()),
                    ("model", Ridge(alpha=alpha)),
                ]
            )
        else:
            model = Pipeline(
                [
                    ("scale", StandardScaler()),
                    ("spline", SplineTransformer(n_knots=6, degree=3, include_bias=False)),
                    ("model", Ridge(alpha=alpha)),
                ]
            )
        model.fit(X_train, y_train_logit)
        return model, {"target_scale": "logit", "alpha": alpha}

    if model_name == "GBR":
        model = GradientBoostingRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.85,
            random_state=seed,
        )
        model.fit(X_train, y_train_logit)
        return model, {"target_scale": "logit", "alpha": np.nan}

    if model_name == "MLP":
        model = Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    MLPRegressor(
                        hidden_layer_sizes=(64,),
                        activation="relu",
                        early_stopping=True,
                        validation_fraction=0.15,
                        max_iter=600,
                        learning_rate_init=1e-3,
                        random_state=seed,
                    ),
                ),
            ]
        )
        model.fit(X_train, y_train_logit)
        return model, {"target_scale": "logit", "alpha": np.nan}

    raise ValueError(f"Unknown model: {model_name}")


def predict_estimator(model, X: np.ndarray, model_info: dict) -> np.ndarray:
    raw = model.predict(X)
    if model_info.get("target_scale") == "normalized":
        return np.clip(raw, 0.0, 1.0)
    return sigmoid(raw)


def grouped_oof_predict(
    meta: pd.DataFrame,
    X: np.ndarray,
    y_norm: np.ndarray,
    model_name: str,
    *,
    n_splits: int = OUTER_FOLDS,
    seed: int = RANDOM_SEED,
    fixed_alpha: float | None = None,
) -> tuple[pd.DataFrame, list[float]]:
    """Grouped out-of-fold predictions for one policy-feature-model combination."""
    groups = meta["run_id"].to_numpy()
    strata = meta["vehicle"].astype(str).to_numpy()
    folds = make_group_stratified_folds(groups, strata, n_splits=n_splits, seed=seed)

    pred = np.full(len(y_norm), np.nan)
    fold_id = np.full(len(y_norm), -1, dtype=int)
    alphas = []

    for k, (tr, te) in enumerate(folds):
        model, info = fit_estimator(
            model_name,
            X[tr],
            y_norm[tr],
            groups_train=groups[tr],
            strata_train=strata[tr],
            seed=seed + 101 * k,
            fixed_alpha=fixed_alpha,
        )
        pred[te] = predict_estimator(model, X[te], info)
        fold_id[te] = k
        alphas.append(info.get("alpha", np.nan))

    pred_df = meta[
        [
            "policy",
            "vehicle",
            "car_name",
            "run_id",
            "round",
            "M",
            "v",
            "margin",
            "B",
            "S",
            "B_norm",
            "S_norm",
            "S_lag1",
            "B_lag1",
            "A",
        ]
    ].copy()
    pred_df["y_true"] = y_norm
    pred_df["y_pred"] = pred
    pred_df["fold"] = fold_id
    return pred_df, alphas


def cv_metrics(pred_df: pd.DataFrame) -> dict:
    y = pred_df["y_true"].to_numpy(dtype=float)
    p = pred_df["y_pred"].to_numpy(dtype=float)
    denom = pred_df["margin"].to_numpy(dtype=float)
    err = p - y

    fold_mae = pred_df.groupby("fold", observed=True).apply(
        lambda d: np.mean(np.abs(d["y_pred"].to_numpy() - d["y_true"].to_numpy())),
        include_groups=False,
    )
    return {
        "n": int(len(pred_df)),
        "n_groups": int(pred_df["run_id"].nunique()),
        "mae_norm": float(np.mean(np.abs(err))),
        "mae_norm_se": float(fold_mae.std(ddof=1) / math.sqrt(len(fold_mae))) if len(fold_mae) > 1 else np.nan,
        "mae_dollar": float(np.mean(np.abs(err) * denom)),
        "rmse_norm": float(np.sqrt(np.mean(err**2))),
        "mape": float(np.mean(np.abs(err) / (np.abs(y) + ETA))),
        "bias": float(np.mean(err)),
        "r2": float(r2_score(y, p)),
    }


def summarize_alpha(alphas: list[float]) -> float:
    vals = np.array([a for a in alphas if np.isfinite(a)], dtype=float)
    if len(vals) == 0:
        return np.nan
    return float(np.median(vals))

## 7. Run the candidate model grid

In [ ]:
FEATURES_TO_RUN = list(FEATURE_SPECS.keys()) if FULL_APPENDIX_RUN else SMOKE_FEATURES
MODELS_TO_RUN = ["Base", "OLS", "Ridge", "GAM", "GBR"]
if INCLUDE_MLP:
    MODELS_TO_RUN.append("MLP")
if not FULL_APPENDIX_RUN:
    MODELS_TO_RUN = SMOKE_MODELS

print("Features:", FEATURES_TO_RUN)
print("Models:", MODELS_TO_RUN)
print(f"Outer folds: {OUTER_FOLDS}, inner folds: {INNER_FOLDS}")

grid_records = []
oof_cache: dict[tuple[str, str, str], pd.DataFrame] = {}

for policy in POLICY_ORDER:
    # Baseline is evaluated once per policy.
    if "Base" in MODELS_TO_RUN:
        meta, X, y, cols = build_model_frame(lagged_turns, policy, "Base")
        pred_df, alphas = grouped_oof_predict(meta, X, y, "Base", n_splits=OUTER_FOLDS, seed=RANDOM_SEED)
        metrics = cv_metrics(pred_df)
        grid_records.append(
            {
                "policy": policy,
                "feature": "Base",
                "model": "Base",
                "lag_order": BASELINE_SPEC["lag_order"],
                "dim_phi": len(BASELINE_SPEC["cols"]) + 1,
                "feature_cols": ",".join(cols),
                "alpha_median": summarize_alpha(alphas),
                **metrics,
            }
        )
        oof_cache[(policy, "Base", "Base")] = pred_df
        print(f"{policy:5s} Base Base    MAE={metrics['mae_norm']:.4f}  R2={metrics['r2']:+.3f}  $MAE={metrics['mae_dollar']:.1f}")

    for feature in FEATURES_TO_RUN:
        meta, X, y, cols = build_model_frame(lagged_turns, policy, feature)

        for model_name in MODELS_TO_RUN:
            if model_name == "Base":
                continue

            try:
                pred_df, alphas = grouped_oof_predict(
                    meta,
                    X,
                    y,
                    model_name,
                    n_splits=OUTER_FOLDS,
                    seed=RANDOM_SEED,
                )
                metrics = cv_metrics(pred_df)
                alpha_median = summarize_alpha(alphas)
                oof_cache[(policy, feature, model_name)] = pred_df
                error = ""
            except Exception as e:
                metrics = {
                    "n": int(len(meta)),
                    "n_groups": int(meta["run_id"].nunique()) if len(meta) else 0,
                    "mae_norm": np.nan,
                    "mae_norm_se": np.nan,
                    "mae_dollar": np.nan,
                    "rmse_norm": np.nan,
                    "mape": np.nan,
                    "bias": np.nan,
                    "r2": np.nan,
                }
                alpha_median = np.nan
                error = f"{type(e).__name__}: {e}"

            spec = FEATURE_SPECS[feature]
            grid_records.append(
                {
                    "policy": policy,
                    "feature": feature,
                    "model": model_name,
                    "lag_order": spec["lag_order"],
                    "dim_phi": len(spec["cols"]) + 1,
                    "feature_cols": ",".join(cols),
                    "alpha_median": alpha_median,
                    **metrics,
                    "error": error,
                }
            )
            print(
                f"{policy:5s} {feature:5s} {model_name:6s} "
                f"MAE={metrics['mae_norm']:.4f}  "
                f"R2={metrics['r2']:+.3f}  "
                f"$MAE={metrics['mae_dollar']:.1f}"
                + (f"  ERR={error}" if error else "")
            )

model_grid = pd.DataFrame(grid_records)
model_grid.to_csv(TABLE_DIR / "B1_full_model_grid.csv", index=False)
display(model_grid.sort_values(["policy", "mae_norm"]).head(20))
print("wrote", TABLE_DIR / "B1_full_model_grid.csv")

## 8. Stage-1 and Stage-2 model selection

Stage 1 keeps candidates whose normalized MAE lies within one standard error of the best candidate for that policy. Stage 2 chooses a linear candidate (OLS or Ridge) from that retained set. If the one-standard-error set contains no linear candidate, the notebook records this and falls back to the best linear model for that policy, which is useful diagnostic information rather than a silent failure.

In [ ]:
def fit_final_linear_for_l1(policy: str, feature: str, model_name: str, alpha: float | None = None) -> dict:
    """Fit a final OLS/Ridge model and report the standardized coefficient L1 norm."""
    meta, X, y, cols = build_model_frame(lagged_turns, policy, feature)
    groups = meta["run_id"].to_numpy()
    strata = meta["vehicle"].astype(str).to_numpy()

    if model_name == "OLS":
        model, info = fit_estimator("OLS", X, y, groups_train=groups, strata_train=strata)
    elif model_name == "Ridge":
        model, info = fit_estimator(
            "Ridge",
            X,
            y,
            groups_train=groups,
            strata_train=strata,
            fixed_alpha=None if alpha is None or not np.isfinite(alpha) else float(alpha),
        )
    else:
        raise ValueError(model_name)

    # Coefficients are on the standardized feature scale because the estimator pipeline scales inputs.
    coef = np.asarray(model.named_steps["model"].coef_, dtype=float)
    intercept = float(model.named_steps["model"].intercept_)
    return {
        "coef_l1": float(np.sum(np.abs(coef))),
        "final_alpha": info.get("alpha", np.nan),
        "intercept_logit": intercept,
        "coef_vector": coef,
        "feature_names": cols,
    }


linear_l1_records = []
for row in model_grid.loc[model_grid["model"].isin(["OLS", "Ridge"])].itertuples(index=False):
    if pd.isna(row.mae_norm):
        continue
    info = fit_final_linear_for_l1(row.policy, row.feature, row.model, row.alpha_median)
    linear_l1_records.append(
        {
            "policy": row.policy,
            "feature": row.feature,
            "model": row.model,
            "coef_l1": info["coef_l1"],
            "final_alpha": info["final_alpha"],
        }
    )

linear_l1 = pd.DataFrame(linear_l1_records)
model_grid = model_grid.merge(linear_l1, on=["policy", "feature", "model"], how="left")
model_grid.to_csv(TABLE_DIR / "B1_full_model_grid_with_l1.csv", index=False)


selection_records = []
for policy, g in model_grid.dropna(subset=["mae_norm"]).groupby("policy", observed=True):
    g = g.copy()
    best = g.loc[g["mae_norm"].idxmin()]
    threshold = best["mae_norm"] + best["mae_norm_se"]

    one_se = g.loc[g["mae_norm"] <= threshold].copy()
    linear_one_se = one_se.loc[one_se["model"].isin(["OLS", "Ridge"])].copy()

    if len(linear_one_se) > 0:
        chosen_pool = linear_one_se.sort_values(["dim_phi", "lag_order", "coef_l1", "mae_norm"], na_position="last")
        within_one_se = True
    else:
        # Diagnostic fallback: no linear model was competitive under the one-SE threshold.
        chosen_pool = g.loc[g["model"].isin(["OLS", "Ridge"])].sort_values(
            ["mae_norm", "dim_phi", "lag_order", "coef_l1"],
            na_position="last",
        )
        within_one_se = False

    chosen = chosen_pool.iloc[0]
    selection_records.append(
        {
            "policy": policy,
            "best_feature_any": best["feature"],
            "best_model_any": best["model"],
            "best_mae_norm": best["mae_norm"],
            "best_mae_norm_se": best["mae_norm_se"],
            "one_se_threshold": threshold,
            "selected_feature": chosen["feature"],
            "selected_model": chosen["model"],
            "selected_mae_norm": chosen["mae_norm"],
            "selected_mae_dollar": chosen["mae_dollar"],
            "selected_r2": chosen["r2"],
            "selected_dim_phi": chosen["dim_phi"],
            "selected_lag_order": chosen["lag_order"],
            "selected_coef_l1": chosen.get("coef_l1", np.nan),
            "linear_within_one_se": within_one_se,
        }
    )

selected_models = pd.DataFrame(selection_records).sort_values("policy")
selected_models.to_csv(TABLE_DIR / "B1_selected_linear_models.csv", index=False)

display(selected_models)
print("wrote", TABLE_DIR / "B1_selected_linear_models.csv")

## 9. Compact summary tables and model-comparison figures

In [ ]:
# Best row per policy-model, regardless of feature map.
best_by_policy_model = (
    model_grid.dropna(subset=["mae_norm"])
    .sort_values(["policy", "model", "mae_norm"])
    .groupby(["policy", "model"], observed=True)
    .head(1)
    .loc[:, ["policy", "model", "feature", "mae_norm", "mae_norm_se", "mae_dollar", "rmse_norm", "r2", "n"]]
    .sort_values(["policy", "mae_norm"])
)
best_by_policy_model.to_csv(TABLE_DIR / "B1_best_feature_by_policy_model.csv", index=False)

# Pivoted MAE table for the manuscript.
mae_pivot = best_by_policy_model.pivot(index="policy", columns="model", values="mae_norm").reindex(POLICY_ORDER)
mae_pivot.to_csv(TABLE_DIR / "B1_mae_pivot_best_feature_by_model.csv")
mae_pivot.to_latex(TABLE_DIR / "B1_mae_pivot_best_feature_by_model.tex", float_format="%.4f")

display(best_by_policy_model)
display(mae_pivot.round(4))

fig, ax = plt.subplots(figsize=(8.0, 4.0))
model_order = [m for m in ["Base", "OLS", "Ridge", "GAM", "GBR", "MLP"] if m in best_by_policy_model["model"].unique()]
for i, model_name in enumerate(model_order):
    vals = best_by_policy_model.loc[best_by_policy_model["model"].eq(model_name), "mae_norm"].dropna().to_numpy()
    ax.scatter(np.full(len(vals), i), vals, alpha=0.65)
    if len(vals):
        ax.scatter([i], [vals.mean()], marker="_", s=900, color="black", zorder=3)
ax.set_xticks(range(len(model_order)))
ax.set_xticklabels(model_order, rotation=30, ha="right")
ax.set_ylabel("Grouped-CV normalized MAE")
ax.set_title("Model comparison: each dot is one policy; black bar is policy average")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_model_comparison_dotplot.png", dpi=200)
plt.show()

print("wrote", FIGURE_DIR / "B1_model_comparison_dotplot.png")

## 10. Reader-visible diagnostics for selected linear models

In [ ]:
selected_oof = {}

for row in selected_models.itertuples(index=False):
    policy = row.policy
    feature = row.selected_feature
    model_name = row.selected_model

    meta, X, y, cols = build_model_frame(lagged_turns, policy, feature)
    alpha = None
    candidate = model_grid.loc[
        (model_grid["policy"].eq(policy))
        & (model_grid["feature"].eq(feature))
        & (model_grid["model"].eq(model_name))
    ]
    if len(candidate) and model_name == "Ridge":
        alpha_val = candidate.iloc[0].get("alpha_median", np.nan)
        alpha = float(alpha_val) if np.isfinite(alpha_val) else None

    pred_df, _ = grouped_oof_predict(meta, X, y, model_name, n_splits=OUTER_FOLDS, fixed_alpha=alpha)
    selected_oof[policy] = pred_df
    pred_df.to_csv(PRED_DIR / f"B1_oof_selected_{policy}.csv", index=False)

    m = cv_metrics(pred_df)
    print(
        f"{policy:5s} {feature:>4s} {model_name:>5s} "
        f"MAE={m['mae_norm']:.4f}, $MAE={m['mae_dollar']:.1f}, R2={m['r2']:+.3f}"
    )

# Predicted vs actual scatter.
fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True)
for ax, policy in zip(axes.flat, POLICY_ORDER):
    dfp = selected_oof[policy]
    ax.scatter(dfp["y_true"], dfp["y_pred"], s=8, alpha=0.25, edgecolor="none")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="black")
    ax.set_title(policy)
    ax.set_xlabel("actual $\\tilde S_t$")
    ax.set_ylabel("predicted $\\hat{\\tilde S}_t$")
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
fig.suptitle("Selected linear models: predicted vs actual seller proposal", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_selected_linear_pred_vs_actual.png", dpi=200)
plt.show()

# Residual histograms.
fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharex=True)
for ax, policy in zip(axes.flat, POLICY_ORDER):
    dfp = selected_oof[policy]
    resid = dfp["y_pred"] - dfp["y_true"]
    ax.hist(resid, bins=30, alpha=0.85)
    ax.axvline(0, linewidth=1, color="black")
    ax.set_title(policy)
    ax.set_xlabel("prediction residual")
    ax.set_ylabel("count")
fig.suptitle("Selected linear models: residual histograms", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_selected_linear_residual_hist.png", dpi=200)
plt.show()

# Binned calibration.
fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True)
for ax, policy in zip(axes.flat, POLICY_ORDER):
    dfp = selected_oof[policy].copy()
    bins = np.unique(np.quantile(dfp["y_pred"], np.linspace(0, 1, 11)))
    if len(bins) < 3:
        continue
    dfp["bin"] = np.digitize(dfp["y_pred"], bins, right=False) - 1
    cal = dfp.groupby("bin", observed=True).agg(pred_mean=("y_pred", "mean"), actual_mean=("y_true", "mean"), n=("y_true", "size"))
    cal = cal.loc[cal["n"] >= 10]
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="black")
    ax.plot(cal["pred_mean"], cal["actual_mean"], marker="o")
    ax.set_title(policy)
    ax.set_xlabel("mean predicted")
    ax.set_ylabel("mean actual")
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
fig.suptitle("Selected linear models: binned calibration by prediction decile", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_selected_linear_calibration.png", dpi=200)
plt.show()

print("wrote selected-diagnostic figures to", FIGURE_DIR)

## 11. Coefficients for selected linear models

The coefficients below are on the standardized feature scale and the logit target scale. Standardized coefficients are useful for comparing signs and relative magnitudes within a fitted model.

In [ ]:
coef_rows = []

for row in selected_models.itertuples(index=False):
    policy = row.policy
    feature = row.selected_feature
    model_name = row.selected_model

    meta, X, y, cols = build_model_frame(lagged_turns, policy, feature)
    groups = meta["run_id"].to_numpy()
    strata = meta["vehicle"].astype(str).to_numpy()

    alpha = None
    if model_name == "Ridge":
        candidate = model_grid.loc[
            (model_grid["policy"].eq(policy))
            & (model_grid["feature"].eq(feature))
            & (model_grid["model"].eq(model_name))
        ]
        if len(candidate):
            alpha_val = candidate.iloc[0].get("alpha_median", np.nan)
            alpha = float(alpha_val) if np.isfinite(alpha_val) else None

    model, info = fit_estimator(
        model_name,
        X,
        y,
        groups_train=groups,
        strata_train=strata,
        fixed_alpha=alpha,
    )

    linear_step = model.named_steps["model"]
    coef = np.asarray(linear_step.coef_, dtype=float)
    coef_rows.append({"policy": policy, "feature": feature, "model": model_name, "term": "intercept", "coef": float(linear_step.intercept_), "alpha": info.get("alpha", np.nan)})
    for name, val in zip(cols, coef):
        coef_rows.append({"policy": policy, "feature": feature, "model": model_name, "term": name, "coef": float(val), "alpha": info.get("alpha", np.nan)})

coef_table = pd.DataFrame(coef_rows)
coef_table.to_csv(TABLE_DIR / "B1_selected_linear_coefficients.csv", index=False)
display(coef_table)
print("wrote", TABLE_DIR / "B1_selected_linear_coefficients.csv")

## 12. Validation A: product-level stability

This section asks whether a model learned from a single vehicle resembles the pooled model. For each policy, the selected feature map/model is fit separately on each vehicle and compared to the pooled fit using standardized coefficient cosine similarity and relative \(L_2\) distance. It also reports cross-vehicle prediction error.

In [ ]:
def fit_standardized_linear_coefficients(
    X: np.ndarray,
    y: np.ndarray,
    model_name: str,
    alpha: float | None,
    scaler: StandardScaler,
) -> np.ndarray:
    Xs = scaler.transform(X)
    y_logit = logit_clip(y)
    if model_name == "OLS":
        est = LinearRegression()
    elif model_name == "Ridge":
        est = Ridge(alpha=1.0 if alpha is None or not np.isfinite(alpha) else float(alpha))
    else:
        raise ValueError("Coefficient stability is only defined for OLS/Ridge.")
    est.fit(Xs, y_logit)
    return np.asarray(est.coef_, dtype=float)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    den = np.linalg.norm(a) * np.linalg.norm(b)
    if den == 0:
        return np.nan
    return float(np.dot(a, b) / den)


product_coef_records = []
product_transfer_records = []

for row in selected_models.itertuples(index=False):
    policy = row.policy
    feature = row.selected_feature
    model_name = row.selected_model
    if model_name not in {"OLS", "Ridge"}:
        continue

    meta_all, X_all, y_all, cols = build_model_frame(lagged_turns, policy, feature)
    alpha = None
    if model_name == "Ridge":
        candidate = model_grid.loc[
            (model_grid["policy"].eq(policy))
            & (model_grid["feature"].eq(feature))
            & (model_grid["model"].eq(model_name))
        ]
        if len(candidate):
            a = candidate.iloc[0].get("alpha_median", np.nan)
            alpha = float(a) if np.isfinite(a) else None

    scaler = StandardScaler().fit(X_all)
    pooled_coef = fit_standardized_linear_coefficients(X_all, y_all, model_name, alpha, scaler)

    for train_v in [f"V{i}" for i in range(1, 6)]:
        train_mask = meta_all["vehicle"].astype(str).eq(train_v).to_numpy()
        if train_mask.sum() < 20:
            continue

        coef_v = fit_standardized_linear_coefficients(X_all[train_mask], y_all[train_mask], model_name, alpha, scaler)
        product_coef_records.append(
            {
                "policy": policy,
                "feature": feature,
                "model": model_name,
                "train_vehicle": train_v,
                "coef_cosine_vs_pooled": cosine_similarity(coef_v, pooled_coef),
                "coef_rel_l2_vs_pooled": float(np.linalg.norm(coef_v - pooled_coef) / (np.linalg.norm(pooled_coef) + 1e-12)),
                "n_train": int(train_mask.sum()),
            }
        )

        # Fit a predictive estimator on this vehicle and evaluate on every vehicle.
        train_groups = meta_all.loc[train_mask, "run_id"].to_numpy()
        train_strata = meta_all.loc[train_mask, "vehicle"].astype(str).to_numpy()
        estimator, info = fit_estimator(
            model_name,
            X_all[train_mask],
            y_all[train_mask],
            groups_train=train_groups,
            strata_train=train_strata,
            fixed_alpha=alpha,
        )

        for test_v in [f"V{i}" for i in range(1, 6)]:
            test_mask = meta_all["vehicle"].astype(str).eq(test_v).to_numpy()
            pred = predict_estimator(estimator, X_all[test_mask], info)
            err = pred - y_all[test_mask]
            product_transfer_records.append(
                {
                    "policy": policy,
                    "feature": feature,
                    "model": model_name,
                    "train_vehicle": train_v,
                    "test_vehicle": test_v,
                    "mae_norm": float(np.mean(np.abs(err))),
                    "rmse_norm": float(np.sqrt(np.mean(err**2))),
                    "n_test": int(test_mask.sum()),
                }
            )

product_coef_table = pd.DataFrame(product_coef_records)
product_transfer_table = pd.DataFrame(product_transfer_records)

product_coef_table.to_csv(TABLE_DIR / "B1_product_coefficient_stability.csv", index=False)
product_transfer_table.to_csv(TABLE_DIR / "B1_product_transfer_mae.csv", index=False)

display(product_coef_table.groupby("policy").agg(
    mean_cosine=("coef_cosine_vs_pooled", "mean"),
    min_cosine=("coef_cosine_vs_pooled", "min"),
    mean_rel_l2=("coef_rel_l2_vs_pooled", "mean"),
))
display(product_transfer_table.head())

# Average cross-vehicle transfer heatmap by policy.
avg_transfer = product_transfer_table.groupby(["train_vehicle", "test_vehicle"], observed=True)["mae_norm"].mean().unstack()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(avg_transfer.to_numpy())
ax.set_xticks(range(len(avg_transfer.columns)))
ax.set_yticks(range(len(avg_transfer.index)))
ax.set_xticklabels(avg_transfer.columns)
ax.set_yticklabels(avg_transfer.index)
ax.set_xlabel("test vehicle")
ax.set_ylabel("train vehicle")
ax.set_title("Average cross-vehicle MAE of selected linear models")
for i in range(avg_transfer.shape[0]):
    for j in range(avg_transfer.shape[1]):
        ax.text(j, i, f"{avg_transfer.iloc[i, j]:.3f}", ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_product_transfer_heatmap.png", dpi=200)
plt.show()

## 13. Validation B: time-window stability

This section fits the selected linear model within early/middle/late stabilized windows and compares coefficients to the pooled fit.

In [ ]:
TIME_WINDOWS = OrderedDict(
    [
        ("early_3_5", (3, 5)),
        ("middle_6_8", (6, 8)),
        ("late_9_12", (9, 12)),
    ]
)

time_stability_records = []

for row in selected_models.itertuples(index=False):
    policy = row.policy
    feature = row.selected_feature
    model_name = row.selected_model
    if model_name not in {"OLS", "Ridge"}:
        continue

    meta_all, X_all, y_all, cols = build_model_frame(lagged_turns, policy, feature)
    alpha = None
    if model_name == "Ridge":
        candidate = model_grid.loc[
            (model_grid["policy"].eq(policy))
            & (model_grid["feature"].eq(feature))
            & (model_grid["model"].eq(model_name))
        ]
        if len(candidate):
            a = candidate.iloc[0].get("alpha_median", np.nan)
            alpha = float(a) if np.isfinite(a) else None

    scaler = StandardScaler().fit(X_all)
    pooled_coef = fit_standardized_linear_coefficients(X_all, y_all, model_name, alpha, scaler)

    for window_name, (lo, hi) in TIME_WINDOWS.items():
        mask = meta_all["round"].between(lo, hi).to_numpy()
        if mask.sum() < 20:
            continue

        coef_w = fit_standardized_linear_coefficients(X_all[mask], y_all[mask], model_name, alpha, scaler)
        time_stability_records.append(
            {
                "policy": policy,
                "feature": feature,
                "model": model_name,
                "time_window": window_name,
                "round_min": lo,
                "round_max": hi,
                "n": int(mask.sum()),
                "coef_cosine_vs_pooled": cosine_similarity(coef_w, pooled_coef),
                "coef_rel_l2_vs_pooled": float(np.linalg.norm(coef_w - pooled_coef) / (np.linalg.norm(pooled_coef) + 1e-12)),
            }
        )

time_stability_table = pd.DataFrame(time_stability_records)
time_stability_table.to_csv(TABLE_DIR / "B1_time_window_stability.csv", index=False)
display(time_stability_table)

fig, ax = plt.subplots(figsize=(8, 4))
for policy in POLICY_ORDER:
    sub = time_stability_table.loc[time_stability_table["policy"].eq(policy)]
    if len(sub):
        ax.plot(sub["time_window"], sub["coef_cosine_vs_pooled"], marker="o", label=policy)
ax.axhline(1, linewidth=1, linestyle="--", color="black")
ax.set_ylabel("cosine similarity to pooled coefficients")
ax.set_xlabel("time window")
ax.set_title("Time-window coefficient stability")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_time_window_coefficient_stability.png", dpi=200)
plt.show()

## 14. Validation C: cross-policy transfer

A seller-policy model should predict its own policy better than other prompt-induced policies. This section trains each selected linear model on one policy and evaluates it on the data from every other policy. Diagonal entries can be read together with the out-of-fold selected-model errors from Section 10.

In [ ]:
cross_policy_records = []

selected_lookup = selected_models.set_index("policy").to_dict(orient="index")

for source_policy in POLICY_ORDER:
    sel = selected_lookup[source_policy]
    feature = sel["selected_feature"]
    model_name = sel["selected_model"]

    meta_src, X_src, y_src, cols = build_model_frame(lagged_turns, source_policy, feature)

    alpha = None
    if model_name == "Ridge":
        candidate = model_grid.loc[
            (model_grid["policy"].eq(source_policy))
            & (model_grid["feature"].eq(feature))
            & (model_grid["model"].eq(model_name))
        ]
        if len(candidate):
            a = candidate.iloc[0].get("alpha_median", np.nan)
            alpha = float(a) if np.isfinite(a) else None

    estimator, info = fit_estimator(
        model_name,
        X_src,
        y_src,
        groups_train=meta_src["run_id"].to_numpy(),
        strata_train=meta_src["vehicle"].astype(str).to_numpy(),
        fixed_alpha=alpha,
    )

    for target_policy in POLICY_ORDER:
        meta_tgt, X_tgt, y_tgt, _ = build_model_frame(lagged_turns, target_policy, feature)
        pred = predict_estimator(estimator, X_tgt, info)
        err = pred - y_tgt
        cross_policy_records.append(
            {
                "source_policy": source_policy,
                "target_policy": target_policy,
                "source_feature": feature,
                "source_model": model_name,
                "mae_norm": float(np.mean(np.abs(err))),
                "mae_dollar": float(np.mean(np.abs(err) * meta_tgt["margin"].to_numpy())),
                "rmse_norm": float(np.sqrt(np.mean(err**2))),
                "bias": float(np.mean(err)),
                "n": int(len(y_tgt)),
            }
        )

cross_policy = pd.DataFrame(cross_policy_records)
cross_policy.to_csv(TABLE_DIR / "B1_cross_policy_transfer.csv", index=False)

cross_mae = cross_policy.pivot(index="source_policy", columns="target_policy", values="mae_norm").reindex(index=POLICY_ORDER, columns=POLICY_ORDER)
display(cross_mae.round(4))

fig, ax = plt.subplots(figsize=(6.4, 5.3))
im = ax.imshow(cross_mae.to_numpy())
ax.set_xticks(range(len(POLICY_ORDER)))
ax.set_yticks(range(len(POLICY_ORDER)))
ax.set_xticklabels(POLICY_ORDER)
ax.set_yticklabels(POLICY_ORDER)
ax.set_xlabel("target policy")
ax.set_ylabel("source policy")
ax.set_title("Cross-policy transfer MAE")
for i in range(cross_mae.shape[0]):
    for j in range(cross_mae.shape[1]):
        ax.text(j, i, f"{cross_mae.iloc[i, j]:.3f}", ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "B1_cross_policy_transfer_heatmap.png", dpi=200)
plt.show()

print("wrote", TABLE_DIR / "B1_cross_policy_transfer.csv")

## 15. Optional: external buyer / alternative-LLM evaluation

Use this section when additional result directories become available, for example an LLM-buyer dataset or a GPT-4o seller dataset with the same JSON schema. The cell is written to skip cleanly when no external directories are configured.

In [ ]:
# Add directories here when they are available. Each directory should contain result JSON
# files with the same run/round schema as the uploaded archive.
EXTERNAL_RESULTS_DIRS: dict[str, Path] = {
    # "llm_buyer": HERE / "0423_strategic_buyer" / "results",
    # "gpt4o_seller": HERE / "gpt4o_used_car_results",
}


def load_any_result_turns(results_dir: Path, dataset_label: str) -> pd.DataFrame:
    """Flexible loader for external result dirs. Policy is taken from each round's active seller strategy."""
    rows = []
    for file in sorted(Path(results_dir).glob("*.json")):
        with open(file, "r", encoding="utf-8") as fh:
            data = json.load(fh)
        exp = data.get("experiment", {})
        tag = exp.get("tag", file.stem)
        for run_index, run in enumerate(data.get("runs", [])):
            car = run["car"]
            for rr in run.get("rounds", []):
                policy_key = rr.get("seller_active_strategy")
                rows.append(
                    {
                        "dataset": dataset_label,
                        "source_file": file.name,
                        "experiment_tag": tag,
                        "policy_key": policy_key,
                        "policy": _policy_label_from_key(policy_key),
                        "run_index_in_file": run_index,
                        "run_id": f"{dataset_label}::{tag}::run{run_index:04d}::seed{run.get('rng_seed')}",
                        "rng_seed": run.get("rng_seed"),
                        "car_name": car["car_name"],
                        "M": float(car["market_price_new"]),
                        "v": float(car["dealer_cost"]),
                        "round": int(rr["round"]),
                        "buyer_price_raw": rr.get("buyer_price"),
                        "seller_price_raw": rr.get("seller_price"),
                        "buyer_template_id": rr.get("buyer_template_id"),
                        "seller_active_strategy": rr.get("seller_active_strategy"),
                        "seller_strategy_changed": rr.get("seller_strategy_changed"),
                    }
                )
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    vehicle_map = {name: f"V{i+1}" for i, name in enumerate(fleet_order)}
    df["vehicle"] = df["car_name"].map(vehicle_map)
    df["policy"] = pd.Categorical(df["policy"], categories=POLICY_ORDER, ordered=True)
    df["vehicle"] = pd.Categorical(df["vehicle"], categories=[f"V{i}" for i in range(1, 6)], ordered=True)
    return df


external_records = []

for label, path in EXTERNAL_RESULTS_DIRS.items():
    if not path.exists():
        print(f"Skipping {label}: directory does not exist: {path}")
        continue

    ext_raw = load_any_result_turns(path, label)
    _, ext_lagged, ext_clean = preprocess_turns(ext_raw)

    for policy in POLICY_ORDER:
        if policy not in set(ext_lagged["policy"].dropna().astype(str)):
            continue

        sel = selected_lookup[policy]
        feature = sel["selected_feature"]
        model_name = sel["selected_model"]

        # Train on randomized-buyer data.
        meta_train, X_train, y_train, _ = build_model_frame(lagged_turns, policy, feature)
        alpha = None
        if model_name == "Ridge":
            candidate = model_grid.loc[
                (model_grid["policy"].eq(policy))
                & (model_grid["feature"].eq(feature))
                & (model_grid["model"].eq(model_name))
            ]
            if len(candidate):
                a = candidate.iloc[0].get("alpha_median", np.nan)
                alpha = float(a) if np.isfinite(a) else None

        estimator, info = fit_estimator(
            model_name,
            X_train,
            y_train,
            groups_train=meta_train["run_id"].to_numpy(),
            strata_train=meta_train["vehicle"].astype(str).to_numpy(),
            fixed_alpha=alpha,
        )

        # Evaluate on the external dataset.
        meta_ext, X_ext, y_ext, _ = build_model_frame(ext_lagged, policy, feature)
        if len(y_ext) == 0:
            continue
        pred = predict_estimator(estimator, X_ext, info)
        err = pred - y_ext
        external_records.append(
            {
                "dataset": label,
                "policy": policy,
                "feature": feature,
                "model": model_name,
                "mae_norm": float(np.mean(np.abs(err))),
                "mae_dollar": float(np.mean(np.abs(err) * meta_ext["margin"].to_numpy())),
                "rmse_norm": float(np.sqrt(np.mean(err**2))),
                "bias": float(np.mean(err)),
                "n": int(len(y_ext)),
                "n_runs": int(meta_ext["run_id"].nunique()),
            }
        )

external_eval = pd.DataFrame(external_records)
if len(external_eval):
    external_eval.to_csv(TABLE_DIR / "B1_external_dataset_evaluation.csv", index=False)
    display(external_eval)
else:
    print("No external datasets were configured or available.")

## 16. Final output manifest

In [ ]:
manifest = []
for folder in [TABLE_DIR, FIGURE_DIR, PRED_DIR]:
    for p in sorted(folder.glob("*")):
        manifest.append({"type": folder.name, "file": str(p.relative_to(OUTPUT_DIR)), "bytes": p.stat().st_size})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUTPUT_DIR / "manifest.csv", index=False)
display(manifest_df)
print("All outputs are under:", OUTPUT_DIR.resolve())